# 01 - Backend Foundation (Hue Foods RAG MVP)

Notebook này trình bày backend skeleton Phase 1 của Hue Foods RAG MVP: package layout, central settings, logging setup và shared retrieval schema. Notebook import các backend modules thay vì duplicate runtime logic.

Chạy cells từ repo root hoặc từ `notebooks/`; cell đầu tiên tự resolve đường dẫn `backend/` từ working directory.

In [ ]:
import sys
from pathlib import Path

for base in (Path.cwd(), *Path.cwd().parents):
    if (base / "backend").is_dir():
        sys.path.insert(0, str(base / "backend"))
        break
print(f"backend on path: {sys.path[0]}")

## Package layout

`backend/` là runtime Python của toàn bộ MVP, tổ chức theo từng component của data flow:

- `config/` - configuration trung tâm: `settings.yaml`, `logging.yaml`, `README_config.md` (không phải Python package).
- `core/` - settings loader, logging setup và shared schema.
- `api/`, `ingestion/`, `embedding/`, `vectorstore/`, `retrieval/`, `scoring/`, `reranking/`, `llm/`, `evaluation/` - package markers cho các phase sau.

Mỗi package có `__init__.py` marker. `tests/` chứa pytest suite của Phase 2, không phải package.

In [ ]:
from pathlib import Path

backend_dir = Path(sys.path[0])
packages = sorted(
    p.name for p in backend_dir.iterdir()
    if p.is_dir() and (p / "__init__.py").exists()
)
print(f"python packages: {len(packages)}")
print(", ".join(packages))
print("core modules:", ", ".join(sorted(p.name for p in (backend_dir / "core").glob("*.py"))))

## Settings (`settings.yaml`)

`core/settings_loader.py` đọc `backend/config/settings.yaml` và xác nhận `active_profile` có tồn tại trong `profiles` (fail fast).

Ba retrieval profiles đã chốt:

| Profile | `retrieval_mode` | `use_bm25` | `use_reranker` |
|---|---|---:|---:|
| `dense_only` | `dense` | `false` | `false` |
| `hybrid_no_rerank` | `hybrid` | `true` | `false` |
| `hybrid_rerank` | `hybrid` | `true` | `true` |

Các nhóm config khác (`knowledge_base`, `embedding`, `vector_database`, `retrieval`, `reranking`, `llm`, `evaluation`) sẽ được các phase sau sử dụng.

In [ ]:
from core.settings_loader import load_settings

settings = load_settings()
print("active_profile:", settings["active_profile"])
for name, profile in settings["profiles"].items():
    print(name, profile["retrieval_mode"], "bm25:", profile["use_bm25"], "rerank:", profile["use_reranker"])
print("config groups:", ", ".join(sorted(settings.keys())))

In [ ]:
from core.settings_loader import load_settings, _validate_active_profile

bad = dict(load_settings(), active_profile="not_a_profile")
try:
    _validate_active_profile(bad)
except ValueError as exc:
    print("ValueError raised:", exc)

## Logging (`logging.yaml`)

`core/logging_setup.py` áp dụng cấu hình logging từ `backend/config/logging.yaml`: ghi ra console và file `backend/logs/application.log`. Đường dẫn log file được pin theo `backend/` nên kết quả độc lập với working directory; `logs/` nằm trong `.gitignore`.

Cell dưới đây chạy logging smoke và tự xóa log file tạo ra sau khi kiểm tra (không để lại artifact).

In [ ]:
import logging
from pathlib import Path
from core.logging_setup import setup_logging, LOGS_DIR, LOG_FILE_NAME

setup_logging()
logging.getLogger("retrieval").info("notebook logging smoke")
log_path = LOGS_DIR / LOG_FILE_NAME
print("log file exists:", log_path.exists())
print("log file path:", log_path)
log_path.unlink(missing_ok=True)

## Shared schema (`schema.py`)

`core/schema.py` định nghĩa `RetrievedDocument` - dataclass shared cho các phase retrieval/reranking/context sau này. Phase 1 chỉ định nghĩa schema, chưa tạo retrieval behavior:

- `id`: chunk id
- `score`: điểm số
- `text`: nội dung document
- `metadata`: dict chứa `source`, `title`, `section` và các score field của stage đã chạy

In [ ]:
from core.schema import RetrievedDocument

doc = RetrievedDocument(
    id="foods/restaurants/example.md|Tóm tắt|0",
    score=0.87,
    text="Nội dung section ví dụ.",
    metadata={"source": "foods/restaurants/example.md", "section": "Tóm tắt"},
)
print(doc.id, doc.score, doc.metadata["section"])

## Checklist xác nhận Phase 1

Người dùng tự chạy notebook này và đối chiếu kết quả mong đợi:

1. `backend on path` trỏ đúng thư mục `backend/` của repo.
2. Danh sách Python packages liệt kê đủ package markers; `core/` chứa `settings_loader.py`, `logging_setup.py`, `schema.py`.
3. `active_profile` trả về `dense_only`; cả ba profiles resolve đúng mode/BM25/reranker flags.
4. Profile không hợp lệ ném `ValueError` kèm danh sách profile hợp lệ.
5. Logging ghi console và tạo đúng `backend/logs/application.log` (rồi tự xóa smoke log).
6. `RetrievedDocument` import và khởi tạo được với đủ bốn fields.
7. Không có network, model, web hoặc external service access; không có API key nào được sử dụng.

Không có live model/API/web call nào trong notebook này.